# Ensemble Methods: Combining Models for Better Predictions

## 1. Introduction

What if instead of relying on a single model, we could combine multiple models to make better predictions?

This is the core idea behind **ensemble methods** - techniques that aggregate predictions from multiple models to achieve better performance than any individual model.

### What We'll Learn

In this notebook, we'll build deep intuitions about:

1. **Why ensembles work** - The "wisdom of crowds" and variance reduction
2. **Bagging** (Bootstrap Aggregating) - Training models on different data subsets
3. **Random Forests** - The most popular ensemble method
4. **Boosting** - Sequential learning from mistakes (AdaBoost, Gradient Boosting)
5. **Stacking** - Using a meta-model to combine predictions
6. **When to use ensembles** - Practical considerations

### Why It Matters

Ensemble methods are among the most powerful and practical techniques in machine learning:

- **Kaggle competitions**: Almost every winning solution uses ensembles
- **Production systems**: Random Forests and Gradient Boosting (XGBoost, LightGBM) are workhorses
- **Robustness**: Ensembles are less sensitive to noise and outliers
- **Simple to implement**: Often just "train multiple models and average"

Let's dive in!

## 2. Setup

We'll use scikit-learn for ensemble implementations and build some from scratch to understand the principles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

Let's also create a helper function for visualizing decision boundaries.

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """Plot decision boundary for 2D classification problem."""
    h = 0.02  # step size in the mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolors='black', s=50)
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.grid(True, alpha=0.3)
    plt.show()

## 3. Why Ensembles Work: The Wisdom of Crowds

### The Core Intuition

Imagine you're trying to estimate the number of jellybeans in a jar. You could:
1. Ask one expert and trust their answer
2. Ask 100 people and take the average

Surprisingly, **the average of many guesses often beats the best individual guess**.

Why? Because:
- Individual errors tend to cancel out (some too high, some too low)
- The average captures the "signal" while filtering the "noise"

This is the "**wisdom of crowds**" - and it's exactly how ensembles work!

### Demonstration: Averaging Reduces Variance

Let's simulate the jellybean jar problem to see this effect in action.

In [ ]:
# True number of jellybeans
true_value = 250

# Simulate 100 people guessing (centered around truth but with noise)
guesses = np.random.normal(loc=true_value, scale=50, size=100)

# Calculate running average as we add more guesses
cumulative_avg = np.cumsum(guesses) / np.arange(1, 101)

plt.figure(figsize=(12, 5))

# Plot individual guesses
plt.subplot(1, 2, 1)
plt.scatter(range(100), guesses, alpha=0.5, s=20)
plt.axhline(y=true_value, color='red', linestyle='--', linewidth=2, label='True Value')
plt.xlabel('Person')
plt.ylabel('Guess')
plt.title('Individual Guesses (Noisy!)')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot cumulative average
plt.subplot(1, 2, 2)
plt.plot(range(1, 101), cumulative_avg, linewidth=2)
plt.axhline(y=true_value, color='red', linestyle='--', linewidth=2, label='True Value')
plt.xlabel('Number of People')
plt.ylabel('Average Guess')
plt.title('Cumulative Average (Converges to Truth!)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"True value: {true_value}")
print(f"Best individual guess: {guesses[np.argmin(np.abs(guesses - true_value))]:.1f}")
print(f"Average of all guesses: {np.mean(guesses):.1f}")
print(f"Error of best individual: {abs(guesses[np.argmin(np.abs(guesses - true_value))] - true_value):.1f}")
print(f"Error of average: {abs(np.mean(guesses) - true_value):.1f}")

### Key Insight: Variance Reduction

Notice how:
- Individual guesses vary wildly (high variance)
- The average converges toward the true value as we add more guesses
- The average is typically better than most individual guesses

This same principle applies to machine learning models!

### Mathematical Foundation: Why Averaging Reduces Variance

Suppose we have $n$ models, each with prediction variance $\sigma^2$ and predictions that are uncorrelated.

**Variance of single model**: $\text{Var}(f_i) = \sigma^2$

**Variance of ensemble average**: $\text{Var}\left(\frac{1}{n}\sum_{i=1}^n f_i\right) = \frac{\sigma^2}{n}$

This means:
- 10 models reduce variance by 10x
- 100 models reduce variance by 100x

The key requirement: **models must make different errors** (be diverse).

Let's verify this mathematically with a simulation.

In [ ]:
# Simulate predictions from multiple models
n_models = [1, 2, 5, 10, 25, 50, 100]
n_trials = 1000
variance_single_model = 1.0

variances = []
theoretical_variances = []

for n in n_models:
    # For each trial, simulate n models and compute their average
    ensemble_predictions = []
    
    for _ in range(n_trials):
        # Each model's prediction (mean=0, variance=variance_single_model)
        model_preds = np.random.normal(0, np.sqrt(variance_single_model), size=n)
        ensemble_pred = np.mean(model_preds)
        ensemble_predictions.append(ensemble_pred)
    
    # Empirical variance of ensemble
    empirical_var = np.var(ensemble_predictions)
    variances.append(empirical_var)
    
    # Theoretical variance
    theoretical_var = variance_single_model / n
    theoretical_variances.append(theoretical_var)

plt.figure(figsize=(10, 6))
plt.plot(n_models, variances, 'o-', label='Empirical Variance', linewidth=2, markersize=8)
plt.plot(n_models, theoretical_variances, 's--', label='Theoretical (σ²/n)', linewidth=2, markersize=8)
plt.xlabel('Number of Models in Ensemble')
plt.ylabel('Prediction Variance')
plt.title('Variance Reduction Through Ensembling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.yscale('log')
plt.show()

print("Variance reduction factors:")
for n, var in zip(n_models, variances):
    reduction = variance_single_model / var
    print(f"  {n:3d} models: {reduction:.1f}x reduction")

### The Critical Requirement: Model Diversity

For ensembles to work, models must be **diverse** - they should make different errors.

If all models are identical (perfectly correlated), averaging does nothing:

$$\text{Var}\left(\frac{1}{n}\sum_{i=1}^n f_i\right) = \sigma^2 \quad \text{(no reduction!)}$$

**How to create diverse models:**
1. Train on different subsets of data (Bagging)
2. Use different feature subsets (Random Forests)
3. Use different model architectures (Stacking)
4. Train sequentially to correct errors (Boosting)

## 4. Bagging: Bootstrap Aggregating

### The Core Idea

**Bagging** (Bootstrap Aggregating) creates diverse models by training each on a different **bootstrap sample** of the data.

**Bootstrap sampling**: Randomly sample $n$ examples from the dataset **with replacement**.

**Algorithm:**
1. Create $B$ bootstrap samples from training data
2. Train a model on each bootstrap sample
3. For prediction:
   - **Classification**: Take majority vote
   - **Regression**: Take average

This creates diversity because each model sees slightly different training data.

### Understanding Bootstrap Sampling

Let's visualize what bootstrap sampling looks like.

In [ ]:
# Original dataset (small for visualization)
original_data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Create 3 bootstrap samples
n_samples = len(original_data)
bootstrap_samples = []

for i in range(3):
    # Sample with replacement
    bootstrap = np.random.choice(original_data, size=n_samples, replace=True)
    bootstrap_samples.append(bootstrap)
    print(f"Bootstrap Sample {i+1}: {sorted(bootstrap)}")
    
    # Count unique values
    unique_values = len(np.unique(bootstrap))
    print(f"  → Contains {unique_values}/10 unique values from original data")
    print(f"  → Some values appear multiple times (duplicates)")
    print()

**Key observation**: Each bootstrap sample:
- Has the same size as the original dataset
- Contains ~63% of unique examples (on average)
- Has some examples appearing multiple times
- Is different from other bootstrap samples

This creates natural diversity!

### Bagging in Action: Stabilizing Decision Trees

Decision trees are **high variance** models - small changes in data lead to completely different trees.

Let's see how a single decision tree behaves vs. bagged trees.

In [ ]:
# Create a 2D classification dataset
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

First, let's train a single deep decision tree (prone to overfitting).

In [ ]:
# Single decision tree (deep, will overfit)
single_tree = DecisionTreeClassifier(max_depth=10, random_state=42)
single_tree.fit(X_train, y_train)

train_acc_single = accuracy_score(y_train, single_tree.predict(X_train))
test_acc_single = accuracy_score(y_test, single_tree.predict(X_test))

print(f"Single Tree:")
print(f"  Train Accuracy: {train_acc_single:.3f}")
print(f"  Test Accuracy:  {test_acc_single:.3f}")
print(f"  Overfitting Gap: {train_acc_single - test_acc_single:.3f}")

Now let's try bagging with 50 trees.

In [ ]:
# Bagging with 50 trees
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=10),
    n_estimators=50,
    random_state=42
)
bagging.fit(X_train, y_train)

train_acc_bagging = accuracy_score(y_train, bagging.predict(X_train))
test_acc_bagging = accuracy_score(y_test, bagging.predict(X_test))

print(f"Bagging (50 trees):")
print(f"  Train Accuracy: {train_acc_bagging:.3f}")
print(f"  Test Accuracy:  {test_acc_bagging:.3f}")
print(f"  Overfitting Gap: {train_acc_bagging - test_acc_bagging:.3f}")
print()
print(f"Test Accuracy Improvement: {test_acc_bagging - test_acc_single:.3f}")

Visualize the decision boundaries to see the difference.

In [ ]:
plt.figure(figsize=(14, 5))

# Single tree
plt.subplot(1, 2, 1)
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = single_tree.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
plt.title(f'Single Tree (Test Acc: {test_acc_single:.3f})\nComplex, Jagged Boundary')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

# Bagging
plt.subplot(1, 2, 2)
Z = bagging.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
plt.title(f'Bagging (Test Acc: {test_acc_bagging:.3f})\nSmooth, Generalized Boundary')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

### Key Insight: Smoothing Through Averaging

Notice how:
- **Single tree**: Creates a very jagged, complex decision boundary (overfitting)
- **Bagging**: Produces a smoother boundary by averaging 50 different trees
- **Test accuracy**: Bagging generalizes better to unseen data

Bagging reduces variance by averaging out the individual quirks of each tree!

### Effect of Number of Trees

How many trees do we need? Let's find out.

In [ ]:
n_estimators_range = [1, 2, 5, 10, 25, 50, 100, 200]
train_scores = []
test_scores = []

for n_est in n_estimators_range:
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=10),
        n_estimators=n_est,
        random_state=42
    )
    bag.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, bag.predict(X_train)))
    test_scores.append(accuracy_score(y_test, bag.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, train_scores, 'o-', label='Train Accuracy', linewidth=2, markersize=8)
plt.plot(n_estimators_range, test_scores, 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Bagging: Effect of Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Test accuracy converges around {n_estimators_range[np.argmax(np.array(test_scores) > max(test_scores) - 0.01)]} trees")

**Key observations:**
- Test accuracy improves rapidly with more trees
- Returns diminish after ~50-100 trees
- More trees never hurt (just slower to train/predict)
- Bagging reduces the train-test gap (less overfitting)

## 5. Random Forests: Bagging++

### The Innovation

**Random Forests** improve upon bagging with one additional trick: **random feature selection**.

When building each tree:
1. Use a bootstrap sample of data (like bagging)
2. **At each split**, only consider a random subset of features

**Why this helps**: In bagging, trees can still be similar if a few features are very strong (they all split on those features). Random feature selection **forces diversity** by making trees explore different feature combinations.

### Random Forests in Action

Let's compare bagging vs. random forests on a classification task.

In [ ]:
# Create a more complex dataset with many features
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Execute the training loop.

In [ ]:
# Single tree
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, tree.predict(X_test))

# Bagging (all features at each split)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)
bagging.fit(X_train, y_train)
acc_bagging = accuracy_score(y_test, bagging.predict(X_test))

# Random Forest (sqrt(n_features) per split by default)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))

print("Test Accuracy Comparison:")
print(f"  Single Tree:    {acc_tree:.4f}")
print(f"  Bagging:        {acc_bagging:.4f}  (+{acc_bagging - acc_tree:.4f})")
print(f"  Random Forest:  {acc_rf:.4f}  (+{acc_rf - acc_tree:.4f})")
print()
print(f"Random Forest vs Bagging improvement: +{acc_rf - acc_bagging:.4f}")

### Feature Importance

One major advantage of Random Forests: we can measure **feature importance** - which features matter most for predictions.

In [ ]:
# Get feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5))
plt.bar(range(X.shape[1]), importances[indices])
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.title('Random Forest: Feature Importances')
plt.xticks(range(X.shape[1]), indices)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

print("Top 5 most important features:")
for i in range(5):
    print(f"  Feature {indices[i]}: {importances[indices[i]]:.4f}")

### Hyperparameter: max_features

The key hyperparameter in Random Forests is `max_features` - how many features to consider at each split.

Common choices:
- **sqrt(n_features)**: Default for classification (good balance)
- **log2(n_features)**: More decorrelation (very diverse trees)
- **n_features**: Same as bagging (less diverse)

Let's see the effect.

In [ ]:
n_features = X.shape[1]
max_features_options = [1, 2, int(np.sqrt(n_features)), int(np.log2(n_features)), n_features // 2, n_features]
test_scores_rf = []

for max_feat in max_features_options:
    rf = RandomForestClassifier(n_estimators=100, max_features=max_feat, random_state=42)
    rf.fit(X_train, y_train)
    test_scores_rf.append(accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(max_features_options, test_scores_rf, 'o-', linewidth=2, markersize=10)
plt.axvline(x=int(np.sqrt(n_features)), color='red', linestyle='--', 
            label=f'sqrt(n_features) = {int(np.sqrt(n_features))}')
plt.xlabel('max_features (features per split)')
plt.ylabel('Test Accuracy')
plt.title('Random Forest: Effect of max_features')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Best max_features: {max_features_options[np.argmax(test_scores_rf)]}")
print(f"Default sqrt(n_features): {int(np.sqrt(n_features))}")

**Key insight**: 
- Too few features (1-2): Trees are too weak and diverse (high bias)
- Too many features (n_features): Trees are too similar (high variance)
- sqrt(n_features): Sweet spot - good balance of diversity and strength

## 6. Boosting: Learning from Mistakes

### A Different Philosophy

While bagging trains models **in parallel** (independently), **boosting** trains them **sequentially**:

1. Train a model
2. Find examples it gets wrong
3. Train the next model to focus on those hard examples
4. Repeat

**Intuition**: Like studying for an exam - you focus more on problems you got wrong.

This converts **many weak learners** (models slightly better than random) into a **strong learner**.

### AdaBoost: Adaptive Boosting

**AdaBoost** was the first successful boosting algorithm.

**Algorithm:**
1. Start with equal weights for all examples
2. Train a weak learner
3. **Increase weights** of misclassified examples
4. Train next weak learner (focuses on high-weight examples)
5. Repeat
6. Final prediction: **weighted vote** of all learners

Let's see it in action.

In [ ]:
# Use the moons dataset
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Execute the training loop.

In [ ]:
# Single weak learner (shallow tree)
weak_learner = DecisionTreeClassifier(max_depth=1)  # Decision stump
weak_learner.fit(X_train, y_train)
acc_weak = accuracy_score(y_test, weak_learner.predict(X_test))

# AdaBoost with 50 weak learners
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    random_state=42
)
adaboost.fit(X_train, y_train)
acc_ada = accuracy_score(y_test, adaboost.predict(X_test))

print("Test Accuracy:")
print(f"  Single Weak Learner (depth=1): {acc_weak:.4f}")
print(f"  AdaBoost (50 weak learners):   {acc_ada:.4f}")
print(f"  Improvement: +{acc_ada - acc_weak:.4f}")

Visualize how AdaBoost progressively improves.

In [ ]:
# Track accuracy as we add more estimators
n_estimators_range = range(1, 51)
train_scores_ada = []
test_scores_ada = []

for n_est in n_estimators_range:
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=n_est,
        random_state=42
    )
    ada.fit(X_train, y_train)
    train_scores_ada.append(accuracy_score(y_train, ada.predict(X_train)))
    test_scores_ada.append(accuracy_score(y_test, ada.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, train_scores_ada, label='Train Accuracy', linewidth=2)
plt.plot(n_estimators_range, test_scores_ada, label='Test Accuracy', linewidth=2)
plt.xlabel('Number of Weak Learners')
plt.ylabel('Accuracy')
plt.title('AdaBoost: Progressive Improvement')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Gradient Boosting: The Modern Champion

**Gradient Boosting** is a more general and powerful boosting algorithm.

**Key idea**: Train each new model to predict the **residual errors** (mistakes) of the previous ensemble.

**Algorithm:**
1. Start with a simple prediction (e.g., mean of targets)
2. Calculate residuals: $r_i = y_i - \hat{y}_i$
3. Train a model to predict these residuals
4. Add this model to ensemble: $\hat{y}_{\text{new}} = \hat{y}_{\text{old}} + \alpha \cdot \text{model}(x)$
5. Repeat

The $\alpha$ is called the **learning rate** - smaller values mean more conservative updates (need more trees but generalize better).

In [ ]:
# Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
acc_gb = accuracy_score(y_test, gb.predict(X_test))

# Random Forest for comparison
rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
rf.fit(X_train, y_train)
acc_rf_comp = accuracy_score(y_test, rf.predict(X_test))

print("Test Accuracy Comparison (both with depth=3 trees):")
print(f"  Random Forest:       {acc_rf_comp:.4f}")
print(f"  Gradient Boosting:   {acc_gb:.4f}")
print(f"  Advantage of GB: +{acc_gb - acc_rf_comp:.4f}")

### The Learning Rate Trade-off

Learning rate controls how much each tree contributes. Let's explore the trade-off.

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.5, 1.0]
colors = ['blue', 'green', 'orange', 'red', 'purple']

plt.figure(figsize=(12, 6))

for lr, color in zip(learning_rates, colors):
    test_scores_gb = []
    
    for n_est in range(1, 101):
        gb = GradientBoostingClassifier(
            n_estimators=n_est,
            learning_rate=lr,
            max_depth=3,
            random_state=42
        )
        gb.fit(X_train, y_train)
        test_scores_gb.append(accuracy_score(y_test, gb.predict(X_test)))
    
    plt.plot(range(1, 101), test_scores_gb, label=f'LR={lr}', color=color, linewidth=2)

plt.xlabel('Number of Trees')
plt.ylabel('Test Accuracy')
plt.title('Gradient Boosting: Learning Rate vs Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Key observations:**
- **High learning rate (1.0)**: Fast learning but can overfit or diverge
- **Low learning rate (0.01)**: Needs many trees but often better final accuracy
- **Rule of thumb**: Use low learning rate (0.01-0.1) with many trees (100-1000)

This is similar to learning rates in neural networks!

### Bagging vs Boosting: Visual Comparison

Let's compare the decision boundaries.

In [ ]:
plt.figure(figsize=(15, 5))

models = [
    ('Random Forest\n(Bagging)', rf),
    ('AdaBoost', adaboost),
    ('Gradient Boosting', gb)
]

for idx, (name, model) in enumerate(models, 1):
    plt.subplot(1, 3, idx)
    
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='RdYlBu', 
                edgecolors='black', s=50)
    
    test_acc = accuracy_score(y_test, model.predict(X_test))
    plt.title(f'{name}\nTest Acc: {test_acc:.3f}')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 7. Stacking: Learning to Combine

### The Meta-Learning Approach

All previous methods combine models with simple rules (voting, averaging). **Stacking** uses a **meta-model** to learn how to combine base models.

**Algorithm:**
1. Train multiple diverse base models (e.g., Random Forest, SVM, Logistic Regression)
2. Use their predictions as features
3. Train a meta-model on these "meta-features" to make final prediction

**Key trick**: Use **cross-validated predictions** from base models to avoid overfitting.

### Stacking in Action

Let's create a stacked ensemble with diverse base models.

In [ ]:
# Create a larger dataset for stacking
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=15,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Execute the training loop.

In [ ]:
# Define diverse base models
base_models = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42)),
    ('lr', LogisticRegression(random_state=42, max_iter=1000))
]

# Meta-model (simple logistic regression)
meta_model = LogisticRegression(random_state=42)

# Create stacking ensemble
stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5  # 5-fold cross-validation for meta-features
)

stacking.fit(X_train, y_train)

Compare base models individually vs. stacked.

In [ ]:
# Evaluate each base model individually
print("Base Models (Individual):")
for name, model in base_models:
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"  {name.upper():4s}: {acc:.4f}")

# Evaluate stacking
acc_stacking = accuracy_score(y_test, stacking.predict(X_test))
print(f"\nStacking Ensemble: {acc_stacking:.4f}")
print(f"\nImprovement over best base model: +{acc_stacking - max([accuracy_score(y_test, m.predict(X_test)) for _, m in base_models]):.4f}")

### How Stacking Works: The Meta-Features

Let's peek at what the meta-model sees.

In [ ]:
# Get predictions from base models on test set
meta_features_test = np.column_stack([
    model.predict_proba(X_test)[:, 1] for _, model in base_models
])

print("Example meta-features (base model predictions) for first 5 test examples:")
print("  RF_prob  SVM_prob  LR_prob   → True Label")
for i in range(5):
    print(f"  {meta_features_test[i, 0]:.3f}    {meta_features_test[i, 1]:.3f}     {meta_features_test[i, 2]:.3f}    → {y_test[i]}")

print("\nThe meta-model learns: 'When RF says 0.8 and SVM says 0.3, predict class X'")

### Key Insight: Learning to Weight Experts

Stacking automatically learns:
- Which base models are most reliable
- Which combinations of predictions are informative
- How to resolve disagreements between models

It's like having a team of experts and a manager who learns who to trust in which situations!

## 8. Practical Considerations

### When to Use Which Ensemble Method?

Let's summarize the trade-offs:

#### Bagging (Random Forests)

**Use when:**
- You have high-variance models (deep trees)
- Training can be parallelized (fast!)
- You want robust, out-of-the-box performance
- Interpretability matters (feature importance)

**Pros:**
- Easy to parallelize
- Rarely overfits (can use many trees)
- Works well with default hyperparameters
- Handles missing values and mixed data types

**Cons:**
- Less accurate than boosting on many datasets
- No sequential refinement

**Typical use**: First model to try, production baseline

#### Boosting (Gradient Boosting, XGBoost, LightGBM)

**Use when:**
- You want maximum accuracy
- You can carefully tune hyperparameters
- Sequential training is acceptable
- You're willing to spend time on tuning

**Pros:**
- Often achieves best accuracy
- Handles complex patterns
- Feature importance
- State-of-art implementations (XGBoost, LightGBM, CatBoost)

**Cons:**
- Can overfit if not tuned properly
- Sensitive to hyperparameters
- Cannot parallelize across trees (only within)
- Slower to train

**Typical use**: Kaggle competitions, maximum accuracy needed

#### Stacking

**Use when:**
- You have multiple good models
- You want to squeeze out last few % of accuracy
- Computational cost is not a concern
- You're in a competition setting

**Pros:**
- Can combine very different model types
- Often gives small but consistent improvements
- Flexible (any models, any meta-model)

**Cons:**
- Complex to implement and tune
- Expensive (train many models)
- Risk of overfitting
- Less interpretable

**Typical use**: Final push in competitions, ensembling very different models

### Computational Cost Comparison

Let's measure training time for each method.

In [ ]:
import time

# Create a medium-sized dataset
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Single Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest (100)': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting (100)': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Stacking': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(50, random_state=42)),
            ('lr', LogisticRegression(random_state=42, max_iter=1000))
        ],
        final_estimator=LogisticRegression(random_state=42),
        cv=3
    )
}

print("Training Time Comparison:")
print("=" * 50)

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:25s}: {train_time:6.3f}s  (acc: {acc:.4f})")

**Key takeaway**: There's a trade-off between accuracy and training time. Choose based on your constraints:
- **Speed**: Random Forest (parallelizable)
- **Accuracy**: Gradient Boosting
- **Maximum accuracy**: Stacking (if you have time)

### Ensemble Diversity Matters

Remember: ensembles only help if models are **diverse** (make different errors).

Let's demonstrate this principle.

In [ ]:
# Create dataset
X, y = make_classification(n_samples=500, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scenario 1: Identical models (no diversity)
identical_models = [DecisionTreeClassifier(max_depth=5, random_state=42) for _ in range(10)]
for model in identical_models:
    model.fit(X_train, y_train)

# Predictions are identical!
preds_identical = np.array([model.predict(X_test) for model in identical_models])
ensemble_pred_identical = np.mean(preds_identical, axis=0).round()
acc_identical = accuracy_score(y_test, ensemble_pred_identical)

# Scenario 2: Diverse models (different random seeds)
diverse_models = [DecisionTreeClassifier(max_depth=5, random_state=i) for i in range(10)]
for model in diverse_models:
    model.fit(X_train, y_train)

preds_diverse = np.array([model.predict(X_test) for model in diverse_models])
ensemble_pred_diverse = np.round(np.mean(preds_diverse, axis=0))
acc_diverse = accuracy_score(y_test, ensemble_pred_diverse)

# Measure diversity: average pairwise disagreement
def pairwise_disagreement(predictions):
    """Fraction of examples where two models disagree."""
    n_models = len(predictions)
    disagreements = []
    for i in range(n_models):
        for j in range(i+1, n_models):
            disagreements.append(np.mean(predictions[i] != predictions[j]))
    return np.mean(disagreements)

div_identical = pairwise_disagreement(preds_identical)
div_diverse = pairwise_disagreement(preds_diverse)

print("Diversity and Performance:")
print("=" * 50)
print(f"Identical Models (same seed):")
print(f"  Diversity (disagreement): {div_identical:.4f}")
print(f"  Ensemble Accuracy:        {acc_identical:.4f}")
print()
print(f"Diverse Models (different seeds):")
print(f"  Diversity (disagreement): {div_diverse:.4f}")
print(f"  Ensemble Accuracy:        {acc_diverse:.4f}")
print()
print(f"Improvement from diversity: +{acc_diverse - acc_identical:.4f}")

**Critical insight**: Without diversity, ensembling provides no benefit! This is why:
- Random Forests use bootstrap sampling AND random features
- Boosting focuses on different examples sequentially
- Stacking uses different model architectures

## 9. Key Takeaways

### Core Principles

1. **Wisdom of Crowds**: Averaging many models reduces variance and improves robustness
   - Mathematical basis: $\text{Var}(\text{average}) = \sigma^2 / n$
   - Individual errors cancel out

2. **Diversity is Essential**: Models must make different errors for ensembles to help
   - Create diversity through: data sampling, feature sampling, different architectures
   - Without diversity, ensembling provides no benefit

3. **Two Main Strategies**:
   - **Parallel (Bagging)**: Train independently, average predictions → reduces variance
   - **Sequential (Boosting)**: Train iteratively on mistakes → reduces bias

### Method Summary

| Method | Training | Combination | Best For | Key Hyperparameter |
|--------|----------|-------------|----------|-------------------|
| **Bagging** | Parallel on bootstrap samples | Average/Vote | High variance models | n_estimators |
| **Random Forest** | Bagging + random features | Average/Vote | General purpose, first try | max_features |
| **AdaBoost** | Sequential, reweight errors | Weighted vote | Weak learners | n_estimators |
| **Gradient Boosting** | Sequential on residuals | Weighted sum | Maximum accuracy | learning_rate + n_estimators |
| **Stacking** | Train separately, meta-model | Learned combination | Squeeze last % | meta-model choice |

### Practical Guidelines

**Start simple:**
1. Try Random Forest first (works well out-of-the-box)
2. If you need more accuracy, try Gradient Boosting (tune carefully!)
3. For competitions, consider stacking diverse models

**Hyperparameter priorities:**
- Random Forest: `n_estimators` (100-500), `max_features` (sqrt)
- Gradient Boosting: `learning_rate` (0.01-0.1) + `n_estimators` (100-1000), `max_depth` (3-5)

**Remember:**
- More trees help (until plateau) and rarely hurt
- Bagging: can parallelize, hard to overfit
- Boosting: sequential, can overfit if not tuned
- Diversity > number of models

### The Bottom Line

Ensembles are one of the most reliable ways to improve model performance:
- **Easy to implement**: Often just "train multiple models and average"
- **Robust**: Less sensitive to noise and outliers
- **Proven**: Dominate ML competitions and production systems
- **Universal**: Work with any base model

The key insight: **Many good-enough models beat one perfect model** (which doesn't exist anyway!).